# pyKT data preprocessing

This notebook prepares the MIAAM/NeurIPS dataset for Knowledge Tracing experiments with pyKT. It produces `miaam_data.txt`, a student-sequence file in pyKT's expected format.

## 1) Load MIAAM dataset

Loads the three source files from `../HF_neurips_official_dataset/`:
- `maths_data_filtered.parquet` — 36,837 students, 6,656,038 attempts (students with <5 attempts or adaptive-test-only histories excluded)
- `maths_exercises_table.parquet` — exercise metadata including hierarchy IDs (`objective_id`, `activity_id`, `module_id`) and text content
- `maths_dependencies.json` — activity-level pedagogical dependency graph

In [17]:
import polars as pl
import json
import pathlib
import os

# DATASET = pathlib.Path("../GAIMHE/Neurips")
DATASET = pathlib.Path("../../Neurips")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")
exercises    = pl.read_parquet(DATASET / "data/maths_exercises_table.parquet")
deps         = json.loads((DATASET / "data/maths_dependencies.json").read_text())

print(interactions.shape, exercises.shape)

(6481693, 14) (7151, 12)


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [18]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

missing_screenshots = exercises.filter(~pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"Exercises without a raw screenshot: {len(missing_screenshots)}")
print(missing_screenshots.select(["exercise_id", "source", "module_name", "activity_name"]))

exercises    = exercises.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"\nAfter filtering: {interactions.shape[0]} attempts, {interactions['exercise_id'].n_unique()} unique exercises")

Exercises without a raw screenshot: 12
shape: (12, 4)
┌─────────────────────────────────┬────────┬───────────────────┬─────────────────────────────────┐
│ exercise_id                     ┆ source ┆ module_name       ┆ activity_name                   │
│ ---                             ┆ ---    ┆ ---               ┆ ---                             │
│ str                             ┆ str    ┆ str               ┆ str                             │
╞═════════════════════════════════╪════════╪═══════════════════╪═════════════════════════════════╡
│ a84300cf-53ce-497d-aa20-1778f1… ┆ am     ┆ Nombres et calcul ┆ Table de 5                      │
│ fa2d0f28-0e2c-4d85-a1ef-0df785… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ 87fd4c76-c5dc-4cc4-b056-7096c5… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ e5fae336-e143-4070-b7f3-4ce23a… ┆ am     ┆ Nombres et calcul ┆ Table de 7                      │
│ 415808bf-9327-4e3a-a6af-694cf6… ┆ am     ┆ Nombres et

## 2) Construct pyKT dataset

Enriches the attempt log with integer-encoded IDs and restructures it into per-student sequences for KT model input.

### Join exercise metadata

Join `activity_id` from the exercise table onto the attempt log using `exercise_id` as the key.

In [19]:
interactions = interactions.join(
    exercises.select(["exercise_id", "activity_id"]),
    on="exercise_id",
    how="left",
)

print(interactions.select(["exercise_id", "activity_id"]).head())

shape: (5, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ exercise_id                     ┆ activity_id                     │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 07144c8b-67d5-4596-b9e3-d49841… │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
└─────────────────────────────────┴─────────────────────────────────┘


### Integer-encode IDs

KT models require contiguous integer indices for students, exercises, and skills (for which we use activities). Each UUID is mapped to a 0-based integer by sorting unique values alphabetically and assigning row indices — producing a stable mapping independent of the order of the interaction data.

In [20]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)

interactions = interactions.join(user_id_map, on="user_id", how="left")

print(f"{interactions['user_id_int'].n_unique()} unique students")
print(interactions.select(["user_id", "user_id_int"]).head())

38520 unique students
shape: (5, 2)
┌─────────────────────────────────┬─────────────┐
│ user_id                         ┆ user_id_int │
│ ---                             ┆ ---         │
│ str                             ┆ u32         │
╞═════════════════════════════════╪═════════════╡
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
└─────────────────────────────────┴─────────────┘


In [21]:
exercise_id_map = (
    interactions.select("exercise_id")
    .unique()
    .sort("exercise_id")
    .with_row_index("exercise_id_int")
)

interactions = interactions.join(exercise_id_map, on="exercise_id", how="left")

print(f"{interactions['exercise_id_int'].n_unique()} unique exercises")
print(interactions.select(["exercise_id", "exercise_id_int"]).head())

6936 unique exercises
shape: (5, 2)
┌─────────────────────────────────┬─────────────────┐
│ exercise_id                     ┆ exercise_id_int │
│ ---                             ┆ ---             │
│ str                             ┆ u32             │
╞═════════════════════════════════╪═════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 4022            │
│ f90ab87b-7054-4de8-b836-300653… ┆ 6723            │
│ f90ab87b-7054-4de8-b836-300653… ┆ 6723            │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 1033            │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 1033            │
└─────────────────────────────────┴─────────────────┘


In [22]:
activity_id_map = (
    interactions.select("activity_id")
    .unique()
    .sort("activity_id")
    .with_row_index("activity_id_int")
)

interactions = interactions.join(activity_id_map, on="activity_id", how="left")

print(f"{interactions['activity_id_int'].n_unique()} unique activities")
print(interactions.select(["exercise_id", "activity_id", "activity_id_int"]).head())

351 unique activities
shape: (5, 3)
┌─────────────────────────────────┬─────────────────────────────────┬─────────────────┐
│ exercise_id                     ┆ activity_id                     ┆ activity_id_int │
│ ---                             ┆ ---                             ┆ ---             │
│ str                             ┆ str                             ┆ u32             │
╞═════════════════════════════════╪═════════════════════════════════╪═════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 07144c8b-67d5-4596-b9e3-d49841… ┆ 6               │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
└─────────────────────────────────┴─────────────────────────────────┴───────────────

### Build per-student sequences

Group the enriched attempt log by student and collect each field into a parallel list ordered chronologically by `created_at` (converted to a Unix timestamp in seconds). The result is a dict keyed by `user_id_int`, where each value is a dict of lists — one list per field, all of the same length.

In [23]:
student_sequences = {
    row["user_id_int"]: {
        "exercise_id_int":      [a["exercise_id_int"]      for a in row["attempts"]],
        "activity_id_int":      [a["activity_id_int"]      for a in row["attempts"]],
        "data_correct":         [int(a["data_correct"])     for a in row["attempts"]],
        "created_at":           [a["created_at"]           for a in row["attempts"]],
        "data_duration":        [int(a["data_duration"])    for a in row["attempts"]],
    }
    for row in (
        interactions
        .select(["user_id_int", "exercise_id_int", "activity_id_int", "data_correct", "created_at", "data_duration"])
        .with_columns(pl.col("created_at").dt.epoch(time_unit="s").alias("created_at"))
        .sort(["user_id_int", "created_at"])
        .group_by("user_id_int", maintain_order=True)
        .agg(pl.struct(["exercise_id_int", "activity_id_int", "data_correct", "created_at", "data_duration"]).alias("attempts"))
        .iter_rows(named=True)
    )
}

In [24]:
print(f"{len(student_sequences)} students")
uid = next(iter(student_sequences))
print(f"Example — student {uid}:")
print({k: v[:2] for k, v in student_sequences[uid].items()})

38520 students
Example — student 0:
{'exercise_id_int': [3197, 4699], 'activity_id_int': [22, 22], 'data_correct': [1, 1], 'created_at': [1697417014, 1697417022], 'data_duration': [6280, 3825]}


### Export

Write `miaam_data.txt` in pyKT's expected format: one student per block, with the sequence length on the first line followed by one comma-separated line per field.

In [25]:
if not os.path.exists("../miaam_pykt_data"):
    os.makedirs("../miaam_pykt_data")
    
with open("../miaam_pykt_data/data.txt", "w") as file:
    for student_id, student_history in student_sequences.items():
        file.write(f"{student_id},{len(student_history['exercise_id_int'])}\n")
        for _v in student_history.values():
            file.write(f"{','.join(map(str, _v))}\n")

## 3) KC graph building

Builds the MIAAM pedagogical dependency graph for use by graph-aware KT models (e.g. GKT). Nodes are modules and activities; edges connect each module directly to all its activities (containment) and connect activities to one another via `unlocks_activity_ids` (directed unlock).

The final output is an N×N float32 adjacency matrix saved to `miaam_kc_graph.npz`, indexed by `activity_id_int` from section 2. Activities present in interactions but absent from the dependency graph contribute all-zero rows and columns.

In [26]:
import networkx as nx

G = nx.DiGraph()

# First pass: add all nodes so cross-module unlock edges resolve correctly
for mod_id, mod in deps["modules"].items():
    G.add_node(mod_id, type="module")
    for obj in mod["objectives"].values():
        for act_id in obj["activities"]:
            G.add_node(act_id, type="activity")

# Second pass: add containment and unlock edges
for mod_id, mod in deps["modules"].items():
    for obj in mod["objectives"].values():
        for act_id, act in obj["activities"].items():
            G.add_edge(mod_id, act_id)
            for unlocked_act_id in act["unlocks_activity_ids"]:
                if G.has_node(unlocked_act_id):
                    G.add_edge(act_id, unlocked_act_id)

n_modules    = sum(1 for _, d in G.nodes(data=True) if d["type"] == "module")
n_activities = sum(1 for _, d in G.nodes(data=True) if d["type"] == "activity")
print(f"Nodes: {G.number_of_nodes()} ({n_modules} modules, {n_activities} activities)")
print(f"Edges: {G.number_of_edges()}")

Nodes: 369 (6 modules, 363 activities)
Edges: 810


### Graph visualization

Interactive Plotly plot of the dependency graph. Node names are resolved from `maths_exercises_table.parquet`; module names are prefixed with their source (`AM:` / `MIA:`). Hover over any node to see its name.

- **Blue nodes (large)** — modules
- **Coral nodes (small)** — activities
- **Grey lines** — module → activity containment edges
- **Orange arrows** — directed unlock edges between activities (source activity unlocks target)

Each weakly connected component is laid out independently with `spring_layout` and then arranged side by side, so the 6 module subgraphs appear as distinct islands rather than overlapping.

In [27]:
import polars as pl
import plotly.graph_objects as go
import numpy as np

id_to_name = {
    **{r["module_id"]: f"{r['source'].upper()}: {r['module_name']}" for r in exercises.select(["module_id", "module_name", "source"]).unique().iter_rows(named=True)},
    **{r["activity_id"]: r["activity_name"] for r in exercises.select(["activity_id", "activity_name"]).unique().iter_rows(named=True)},
}

module_nodes   = [n for n, d in G.nodes(data=True) if d["type"] == "module"]
activity_nodes = [n for n, d in G.nodes(data=True) if d["type"] == "activity"]

# Lay out each weakly connected component separately, then arrange them in a row
pos = {}
offset_x = 0.0
for component in nx.weakly_connected_components(G):
    subgraph = G.subgraph(component)
    sub_pos = nx.spring_layout(subgraph, seed=42)
    xs = [p[0] for p in sub_pos.values()]
    width = max(xs) - min(xs) if len(xs) > 1 else 0
    for node, (x, y) in sub_pos.items():
        pos[node] = (x + offset_x, y)
    offset_x += width + 0.3

# Module-activity edges: plain lines
mod_act_edges = [(u, v) for u, v in G.edges() if G.nodes[u]["type"] == "module"]
ex, ey = [], []
for u, v in mod_act_edges:
    x0, y0 = pos[u]; x1, y1 = pos[v]
    ex += [x0, x1, None]; ey += [y0, y1, None]

edge_trace = go.Scatter(x=ex, y=ey, mode="lines",
                        line=dict(width=1, color="#bbb"), hoverinfo="none")

def node_trace(nodes, color, size):
    return go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers",
        marker=dict(color=color, size=size, line=dict(width=1, color="white")),
        text=[id_to_name.get(n, n) for n in nodes],
        hovertemplate="%{text}<extra></extra>",
    )

fig = go.Figure(
    data=[
        edge_trace,
        node_trace(module_nodes,   color="steelblue", size=20),
        node_trace(activity_nodes, color="coral",     size=10),
    ],
    layout=go.Layout(
        showlegend=False,
        hovermode="closest",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(l=0, r=0, t=0, b=0),
        height=600,
    ),
)

# Activity-activity unlock edges: directed arrows via annotations
act_act_edges = [(u, v) for u, v in G.edges() if G.nodes[u]["type"] == "activity"]
for u, v in act_act_edges:
    x0, y0 = pos[u]; x1, y1 = pos[v]
    fig.add_annotation(
        x=x1, y=y1, ax=x0, ay=y0,
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,
        arrowcolor="#c0714a",
    )

fig.show()

### Export adjacency matrix

Build an N×N float32 adjacency matrix over all activities in `activity_id_int` order. Each entry `adj[i, j] = 1` if activity `i` directly unlocks activity `j`. Activities present in interactions but absent from the dependency graph contribute all-zero rows and columns. Saved to `miaam_kc_graph.npz` alongside `miaam_data.txt`.

In [28]:
import numpy as np

act_uuid_to_int = {
    row["activity_id"]: row["activity_id_int"]
    for row in activity_id_map.filter(pl.col("activity_id").is_not_null()).iter_rows(named=True)
}
n_activities = len(act_uuid_to_int)

adj = np.zeros((n_activities, n_activities), dtype=np.float32)

for u, v in G.edges():
    if G.nodes[u]["type"] == "activity" and G.nodes[v]["type"] == "activity":
        i = act_uuid_to_int.get(u)
        j = act_uuid_to_int.get(v)
        if i is not None and j is not None:
            adj[i, j] = 1.0

np.savez("../miaam_pykt_data/gkt_graph_transition.npz", matrix=adj)
print(f"Adjacency matrix shape: {adj.shape}")
print(f"Non-zero entries: {int(np.count_nonzero(adj))}")

Adjacency matrix shape: (351, 351)
Non-zero entries: 439


## 4) Construct train/test dataset for other baselines

Reads the student IDs from pyKT's split files (`train_valid_quelevel.csv` and `test_quelevel.csv`) and filters `interactions` to produce matching train and test sets. PyKT identifies students by `uid`, which corresponds to `user_id_int` in our encoding.

The output files `interactions_train.parquet` and `interactions_test.parquet` are saved at the project root. They retain all columns from `interactions` (including `user_id_int`, `exercise_id_int`, and `activity_id_int`), making it straightforward to run non-pyKT baselines on exactly the same student splits.

In [29]:
train_uids = pl.read_csv("../pykt-toolkit/data/miaamv1/train_valid_quelevel.csv").select("uid").unique()
test_uids  = pl.read_csv("../pykt-toolkit/data/miaamv1/test_quelevel.csv").select("uid").unique()

interactions_train = interactions.join(train_uids.rename({"uid": "user_id_int"}), on="user_id_int", how="inner")
interactions_test  = interactions.join(test_uids.rename({"uid": "user_id_int"}), on="user_id_int", how="inner")

interactions_train.write_parquet("../interactions_train.parquet")
interactions_test.write_parquet("../interactions_test.parquet")

print(f"Train: {interactions_train['user_id_int'].n_unique()} students, {len(interactions_train)} attempts")
print(f"Test:  {interactions_test['user_id_int'].n_unique()} students, {len(interactions_test)} attempts")
print(f"Total number: {interactions_train['user_id_int'].n_unique() + interactions_test['user_id_int'].n_unique()} students, {len(interactions_train) + len(interactions_test)} attempts")

Train: 30816 students, 5044292 attempts
Test:  7704 students, 1221743 attempts
Total number: 38520 students, 6266035 attempts


In [30]:
for name, df in [("Train", interactions_train), ("Test", interactions_test)]:
    pct = df["data_correct"].mean() * 100
    print(f"{name}: {pct:.2f}% correct")

Train: 78.69% correct
Test: 78.60% correct
